### **Advanced SQL**

<font color="red">File access required:</font> In Colab this notebook requires first uploading files **Cities.csv**, **Countries.csv**, **Players.csv**, and **Teams.csv** using the *Files* feature in the left toolbar. If running the notebook on a local computer, simply ensure these files are in the same workspace as the notebook.

In [1]:
!pip install prettytable==0.7.2
!pip install ipython-sql

  Preparing metadata (setup.py) ... done
  Created wheel for prettytable: filename=prettytable-0.7.2-py3-none-any.whl size=13695 sha256=7929b202697f5b8160485736b722609ffe00e77cbe176a754c70add689c04ea6
  Stored in directory: /root/.cache/pip/wheels/ca/f9/66/1ebeb8cdff2211eebb6fce02957f9e0a9ae3da4b7e65512d1b
Successfully built prettytable
  Attempting uninstall: prettytable
    Found existing installation: prettytable 3.17.0
    Uninstalling prettytable-3.17.0:
      Successfully uninstalled prettytable-3.17.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 21.1 MB/s eta 0:00:00


In [2]:
# Set-up
%load_ext sql
%sql sqlite://
import pandas as pd

In [3]:
# Create database tables from CSV files
with open('Cities.csv') as f: Cities = pd.read_csv(f, index_col=0)
%sql drop table if exists Cities;
%sql --persist Cities
with open('Countries.csv') as f: Countries = pd.read_csv(f, index_col=0)
%sql drop table if exists Countries;
%sql --persist Countries

 * sqlite://
Done.
 * sqlite://
 * sqlite://
Done.
 * sqlite://


'Persisted countries'

#### Look at sample of Cities and Countries tables

In [4]:
%%sql
select * from Cities limit 5

 * sqlite://
Done.


city,country,latitude,longitude,temperature
Aalborg,Denmark,57.03,9.92,7.52
Aberdeen,United Kingdom,57.17,-2.08,8.1
Abisko,Sweden,63.35,18.83,0.2
Adana,Turkey,36.99,35.32,18.67
Albacete,Spain,39.0,-1.87,12.62


In [5]:
%%sql
select * from Countries limit 5

 * sqlite://
Done.


country,population,EU,coastline
Albania,2.9,no,yes
Andorra,0.07,no,no
Austria,8.57,yes,no
Belarus,9.48,no,no
Belgium,11.37,yes,yes


### Duplicates, table variables

*Warm-up: Find all cities in the EU with temperature > 15*

In [6]:
%%sql
select city
from Cities, Countries
where Cities.country = Countries.country
and EU = 'yes' and temperature > 15

 * sqlite://
Done.


city
Algeciras
Athens
Badajoz
Barcelona
Bari
Cartagena
Catania
Cosenza
Granada
Huelva


*Modify previous query to return EU countries that have a city with temperature > 15, remove duplicates*

*Find number of countries that have a city with latitude > 60 (start with country list then fix)*

In [7]:
%%sql
select country
from Cities
where latitude > 60

 * sqlite://
Done.


country
Sweden
Norway
Norway
Finland
Sweden
Finland
Finland
Norway
Finland


*Modify first query to use table variables*

In [8]:
%%sql
select city
from Cities, Countries
where Cities.country = Countries.country
and EU = 'yes' and temperature > 15

 * sqlite://
Done.


city
Algeciras
Athens
Badajoz
Barcelona
Bari
Cartagena
Catania
Cosenza
Granada
Huelva


*Find all pairs of cities with the same longitude; return the city pairs and their (shared) longitude - notice what's wrong and fix it*

In [9]:
%%sql
select C1.city, C2.city, C1.longitude
from Cities C1, Cities C2
where C1.longitude = C2.longitude

 * sqlite://
Done.


city,city_1,longitude
Aalborg,Aalborg,9.92
Aberdeen,Aberdeen,-2.08
Abisko,Abisko,18.83
Adana,Adana,35.32
Albacete,Albacete,-1.87
Algeciras,Algeciras,-5.47
Amiens,Amiens,2.3
Amsterdam,Amsterdam,4.92
Ancona,Ancona,13.5
Andorra,Andorra,1.52


*Find all pairs of cities that are near each other, i.e., longitude and latitude are both less than 0.5 apart; return city pairs*

In [10]:
%%sql
select C1.city, C2.city
from Cities C1, Cities C2
where abs(C1.longitude - C2.longitude) < .5
and abs(C1.latitude - C2.latitude) < .5
and C1.city < C2.city

 * sqlite://
Done.


city,city_1
Adana,Tarsus
Ancona,Sarajevo
Basel,Freiburg
Basel,Mulhouse
Bergamo,Milan
Cartagena,Murcia
Heidelberg,Karlsruhe
Horlivka,Makiyivka


### <font color = 'green'>**Your Turn**</font>

*Find all pairs of cities with the same temperature. Return the city pairs along with their shared temperature. What do you think about the data after seeing the answer?*

In [11]:
%%sql
SELECT C1.city AS city1, C2.city AS city2
FROM Cities C1
INNER JOIN Cities C2
ON ABS(C1.longitude - C2.longitude) < 0.5
AND ABS(C1.latitude - C2.latitude) < 0.5
AND C1.city < C2.city;


 * sqlite://
Done.


city1,city2
Adana,Tarsus
Ancona,Sarajevo
Basel,Freiburg
Basel,Mulhouse
Bergamo,Milan
Cartagena,Murcia
Heidelberg,Karlsruhe
Horlivka,Makiyivka


### Subqueries in Where clause

*Find all countries in the Countries table with no city in the Cities table*

In [12]:
%%sql
select country
from Countries
where not exists (

  select * from Cities
  where Cities.country = Countries.country

)

 * sqlite://
Done.


country
Cyprus
Iceland
Kosovo
Liechtenstein
Luxembourg


*Find countries in the EU that have a city with temperature > 15*

In [13]:
%%sql
select country
from Countries
where EU = 'yes'
and exists (select * from Cities
            where Cities.country = Countries.country
            and temperature > 15)

 * sqlite://
Done.


country
Greece
Italy
Portugal
Spain


*Find number of countries that have a city with latitude > 60 (start with country list)*

In [14]:
%%sql
select country
from Countries
where exists (select * from Cities
              where Cities.country = Countries.country
              and latitude > 60)

 * sqlite://
Done.


country
Finland
Norway
Sweden


*Find the westernmost city; return the city and longitude*

In [15]:
%%sql
select city, longitude
from Cities C1
where not exists (select * from Cities C2
                  where C2.longitude < C1.longitude)

 * sqlite://
Done.


city,longitude
Lisbon,-9.14


*Add easternmost to previous query*

*Westernmost city query using = and min*

In [16]:
%%sql
select city, longitude
from Cities
where longitude = (select min(longitude) from Cities)

 * sqlite://
Done.


city,longitude
Lisbon,-9.14


*Find all cities whose temperature is more than 50% higher than the average; return the city, country, and temperature, ordered by descending temperature*

In [17]:
%%sql
select city, country, temperature
from Cities
where temperature > (select avg(temperature) * 1.5 from Cities)
order by temperature desc

 * sqlite://
Done.


city,country,temperature
Adana,Turkey,18.67
Palermo,Italy,17.9
Athens,Greece,17.41
Algeciras,Spain,17.38
Cartagena,Spain,17.32
Kalamata,Greece,17.3
Marbella,Spain,17.19
Huelva,Spain,17.09
Patras,Greece,16.9
Cosenza,Italy,16.6


*Number of cities in the EU*

In [18]:
%%sql
select count()
from Cities
where country in (select country from Countries where EU = 'yes')

 * sqlite://
Done.


count()
150


*Modify previous query to use "not in"*

*Same query using join instead of subquery*

In [19]:
%%sql
select count()
from Cities, Countries
where Cities.country = Countries.country
and EU = 'yes'

 * sqlite://
Done.


count()
150


*Number of countries with no coastline and a city with longitude < 20*

In [20]:
%%sql
select count()
from Countries
where coastline = 'no'
and exists (select * from Cities where country = Countries.country
            and longitude < 20)

 * sqlite://
Done.


count()
7


*Same query using join instead of subquery (see what's wrong and fix it)*

In [21]:
%%sql
select count()
from Countries, Cities
where Countries.country = Cities.country
and coastline = 'no' and longitude < 20

 * sqlite://
Done.


count()
16


*Find countries in Countries table with no city in Cities table using join instead of subquery (subquery version repeated first)*

In [22]:
%%sql
select country
from Countries
where not exists (select * from Cities
                  where Cities.country = Countries.country)

 * sqlite://
Done.


country
Cyprus
Iceland
Kosovo
Liechtenstein
Luxembourg


In [23]:
%%sql
FILL IN

 * sqlite://
(sqlite3.OperationalError) near "FILL": syntax error
[SQL: FILL IN]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


### <font color = 'green'>**Your Turn**</font>

*Find all cities in a country whose population is < 2; return the city and country. First write the query without a subquery.*

In [24]:
%%sql
SELECT Cities.city, Cities.country
FROM Cities
INNER JOIN Countries ON Cities.country = Countries.country
WHERE Countries.population < 2;

 * sqlite://
Done.


city,country
Andorra,Andorra
Daugavpils,Latvia
Podgorica,Montenegro
Riga,Latvia
Tallinn,Estonia
Tartu,Estonia


*Now write the same query using a subquery instead of a join.*

In [25]:
%%sql
SELECT city, country
FROM Cities
WHERE country IN (
SELECT country
FROM Countries
WHERE population < 2
);

 * sqlite://
Done.


city,country
Andorra,Andorra
Daugavpils,Latvia
Podgorica,Montenegro
Riga,Latvia
Tallinn,Estonia
Tartu,Estonia


*Find all countries with no city having a temperature > 6*

In [26]:
%%sql
SELECT country
FROM Countries c
WHERE NOT EXISTS (
    SELECT 1
    FROM Cities ci
    WHERE ci.country = c.country
      AND ci.temperature > 6
);

 * sqlite://
Done.


country
Cyprus
Estonia
Finland
Iceland
Kosovo
Latvia
Liechtenstein
Luxembourg
Norway


*Now try to write the same query without a subquery; can you?*

In [27]:
%%sql
SELECT c.country
FROM Countries c
LEFT JOIN Cities ci
    ON c.country = ci.country AND ci.temperature > 6
WHERE ci.city IS NULL;

 * sqlite://
Done.


country
Cyprus
Estonia
Finland
Iceland
Kosovo
Latvia
Liechtenstein
Luxembourg
Norway


### Aggregation with Having clause

*Find all countries with average city temperature > 10; return country and average temperature*

In [28]:
%%sql
select country, avg(temperature)
from Cities
group by country
having avg(temperature) > 10

 * sqlite://
Done.


country,avg(temperature)
Albania,15.18
Bulgaria,10.44
Croatia,10.865
France,10.151111111111112
Greece,16.9025
Italy,13.474666666666668
Portugal,14.469999999999999
Spain,14.238333333333332
Turkey,11.726666666666665


*Find all countries with more than 5 cities above latitude 50*

In [29]:
%%sql
select country
from Cities
where latitude > 50
group by country
having count() > 5

 * sqlite://
Done.


country
Belarus
Germany
Poland
Sweden
United Kingdom


*Same query without Having clause*

In [30]:
%%sql
select distinct country
from Cities C1
where 5 < (select count() from Cities C2
           where C1.country=C2.country
           and latitude > 50)

 * sqlite://
Done.


country
United Kingdom
Sweden
Germany
Poland
Belarus


*Which combinations of EU versus non-EU and coastline versus no-coastline have a minimum population greater than 0.5?*

In [31]:
%%sql
select EU, coastline, min(population)
from Countries
group by EU, coastline
having min(population) > 0.5

 * sqlite://
Done.


EU,coastline,min(population)
yes,no,0.58
yes,yes,1.18


*Find all countries with average city temperature more than 50% higher than the overall average; return country and average temperature*

In [32]:
%%sql
select country, avg(temperature)
from Cities
group by country
having avg(temperature) > (select 1.5 * avg(temperature) from Cities)

 * sqlite://
Done.


country,avg(temperature)
Albania,15.18
Greece,16.9025
Portugal,14.469999999999999


### <font color = 'green'>**Your Turn**</font>

*Find all countries whose average city longitude is lower than the overall average longitude, and whose average city latitude is higher than the overall average latitude. Return the countries. Note: Yes, you can use "and" in Having clauses!*

In [33]:
%%sql
SELECT country
FROM Cities
GROUP BY country
HAVING AVG(longitude) < (SELECT AVG(longitude) FROM Cities)
   AND AVG(latitude) > (SELECT AVG(latitude) FROM Cities);

 * sqlite://
Done.


country
Austria
Belgium
Denmark
Germany
Ireland
Netherlands
Norway
United Kingdom


### Subqueries in From and Select clauses

*Find all countries with both cold and warm cities -- at least one city with temperature < 9 and one city with temperature > 14*

In [34]:
%%sql
select distinct C1.country
from Cities C1, Cities C2
where C1.country = C2.country
and C1.temperature < 9 and C2.temperature > 14

 * sqlite://
Done.


country
France
Turkey
Italy


*Modify query to also return count of cold and warm cities (then show without column renaming)*

In [35]:
# (select count() from Cities where country = C1.country and temperature < 9) as numcold,
# (select count() from Cities where country = C1.country and temperature > 14) as numwarm
%%sql
SELECT DISTINCT C1.country,
    (SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature < 9),
    (SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature > 14)
FROM Cities C1
WHERE (SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature < 9) > 0
  AND (SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature > 14) > 0;

 * sqlite://
Done.


country,(SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature < 9),(SELECT COUNT(*) FROM Cities WHERE country = C1.country AND temperature > 14)
Turkey,4,5
France,5,1
Italy,1,7


*Same query using subquery in From clause instead of Select clause*

In [36]:
%%sql
select Cold.country, numcold, numwarm
from (select country, count() as numcold from Cities
      where temperature < 9 group by country) Cold,
     (select country, count() as numwarm from Cities
      where temperature > 14 group by country) Warm
where Cold.country = Warm.country

 * sqlite://
Done.


country,numcold,numwarm
France,5,1
Italy,1,7
Turkey,4,5


### Data modification

*Increase all city temperatures by 10%*

In [37]:
%%sql
update Cities
set temperature = 1.1 * temperature

 * sqlite://
213 rows affected.


[]

In [38]:
%%sql
select avg(temperature) from Cities

 * sqlite://
Done.


avg(temperature)
10.447624413145537


*Increase temperatures another 10% for cities in countries with coastline*

In [39]:
%%sql
update Cities
set temperature = 1.1 * temperature
where country in (select country from Countries
                  where coastline = 'yes')

 * sqlite://
183 rows affected.


[]

*Delete all cities in Turkey*

In [40]:
%%sql
delete from Cities
where country = 'Turkey'

 * sqlite://
24 rows affected.


[]

*Create a new table NonEU containing list of cities (with country) not in the EU*

In [41]:
%%sql
drop table if exists NonEU;
create table NonEU(city, country);
insert into NonEU
  select city, country from cities
  where country in (select country from Countries
                    where EU = 'no');
select * from NonEU

 * sqlite://
Done.
Done.
39 rows affected.
Done.


city,country
Andorra,Andorra
Balti,Moldova
Basel,Switzerland
Belgrade,Serbia
Bergen,Norway
Bila Tserkva,Ukraine
Bodo,Norway
Brest,Belarus
Cherkasy,Ukraine
Chernihiv,Ukraine


*Add your city*

In [42]:
%%sql
insert into NonEU values ('Cavite','Philippines');
select * from NonEU

 * sqlite://
1 rows affected.
Done.


city,country
Andorra,Andorra
Balti,Moldova
Basel,Switzerland
Belgrade,Serbia
Bergen,Norway
Bila Tserkva,Ukraine
Bodo,Norway
Brest,Belarus
Cherkasy,Ukraine
Chernihiv,Ukraine


### <font color = 'green'>**Your Turn - Advanced SQL on World Cup Data**</font>

In [43]:
# Create database tables from CSV files
with open('Players.csv') as f: Players = pd.read_csv(f, index_col=0)
%sql drop table if exists Players;
%sql --persist Players
with open('Teams.csv') as f: Teams = pd.read_csv(f, index_col=0)
%sql drop table if exists Teams;
%sql --persist Teams

 * sqlite://
Done.
 * sqlite://
 * sqlite://
Done.
 * sqlite://


'Persisted teams'

#### Look at sample of Players and Teams tables

In [44]:
%%sql
select * from Players limit 5

 * sqlite://
Done.


surname,team,position,minutes,shots,passes,tackles,saves
Abdoun,Algeria,midfielder,16,0,6,0,0
Belhadj,Algeria,defender,270,1,146,8,0
Boudebouz,Algeria,midfielder,74,3,28,1,0
Bougherra,Algeria,defender,270,1,89,11,0
Chaouchi,Algeria,goalkeeper,90,0,17,0,2


In [45]:
%%sql
select * from Teams limit 5

 * sqlite://
Done.


team,ranking,games,wins,draws,losses,goalsFor,goalsAgainst,yellowCards,redCards
Brazil,1,5,3,1,1,9,4,7,2
Spain,2,6,5,0,1,7,2,3,0
Portugal,3,4,1,2,1,7,1,8,1
Netherlands,4,6,6,0,0,12,5,15,0
Italy,5,3,0,2,1,4,5,5,0


*1) Find all pairs of teams who have the same number of goalsFor as
each other and the same number of goalsAgainst as each other.
Return the teams and numbers of goalsFor and goalsAgainst.
Make sure to return each pair only once.*

In [46]:
%%sql
SELECT Teams.team, Teams_2.team, Teams.goalsFor, Teams.goalsAgainst
FROM Teams
INNER JOIN Teams Teams_2
    ON Teams.goalsFor = Teams_2.goalsFor
   AND Teams.goalsAgainst = Teams_2.goalsAgainst
   AND Teams.team < Teams_2.team;

 * sqlite://
Done.


team,team_1,goalsFor,goalsAgainst
Italy,Mexico,4,5
England,Nigeria,3,5
England,South Africa,3,5
Chile,England,3,5
Chile,Nigeria,3,5
Chile,South Africa,3,5
Cameroon,Greece,2,5
Australia,Denmark,3,6
Nigeria,South Africa,3,5


*2) Find all teams with ranking <30 where no player made more than 150 passes. Return the team and ranking.*

In [47]:
%%sql
SELECT team, ranking
FROM Teams
WHERE ranking < 30
  AND NOT EXISTS (
      SELECT 1
      FROM Players
      WHERE Players.team = Teams.team
        AND Players.passes > 150
  );

 * sqlite://
Done.


team,ranking
France,9
Nigeria,21
Switzerland,24


*3) Which players made more shots than 5x the overall average number of shots? Return the player surname, position, and team.*

In [48]:
%%sql
SELECT surname, position, team
FROM Players
WHERE shots > 5 * (SELECT AVG(shots) FROM Players);

 * sqlite://
Done.


surname,position,team
Higuain,forward,Argentina
Messi,forward,Argentina
Podolski,forward,Germany
Boateng,midfielder,Ghana
Gyan,forward,Ghana
Sneijder,midfielder,Netherlands
van Persie,forward,Netherlands
Jong Tae-Se,forward,North Korea
Ronaldo,forward,Portugal
Park Chu-Young,forward,South Korea


*4) Find all team-position pairs where the average number of passes made by players in that position on that team is greater than 150. Return the team-position pairs.*

In [49]:
%%sql
SELECT team, position
FROM Players
GROUP BY team, position
HAVING AVG(passes) > 150;


 * sqlite://
Done.


team,position
Argentina,midfielder
Brazil,defender
Germany,defender
Germany,midfielder
Ghana,midfielder
Mexico,defender
Netherlands,defender
Netherlands,midfielder
Spain,defender
Spain,midfielder


*5) Find all teams whose defenders averaged more than 150 passes. Return the team and average number of passes by defenders, in descending order of average passes.*

In [50]:
%%sql
SELECT team, AVG(passes)
FROM Players
WHERE position = 'defender'
GROUP BY team
HAVING AVG(passes) > 150
ORDER BY AVG(passes) DESC;

 * sqlite://
Done.


team,AVG(passes)
Spain,213.0
Brazil,190.0
Germany,189.83333333333334
Netherlands,182.5
Mexico,152.14285714285714


### <font color = 'green'>**Your Turn Extra - Advanced SQL on Titanic Data**</font>

<font color="red">File access required:</font> In Colab these extra problems require first uploading **Titanic.csv** using the *Files* feature in the left toolbar. If running the notebook on a local computer, simply ensure this file is in the same workspace as the notebook.

In [51]:
# Load dataabase table from CSV file
with open('Titanic.csv') as f: Titanic = pd.read_csv(f, index_col=0)
%sql drop table if exists Titanic;
%sql --persist Titanic

 * sqlite://
Done.
 * sqlite://


'Persisted titanic'

#### Look at sample of Titanic table

In [52]:
%%sql
select * from Titanic limit 5

 * sqlite://
Done.


last,first,gender,age,class,fare,embarked,survived
Abbing,Mr. Anthony,M,42.0,3,7.55,Southampton,no
Abbott,Mrs. Stanton (Rosa Hunt),F,35.0,3,20.25,Southampton,yes
Abbott,Mr. Rossmore Edward,M,16.0,3,20.25,Southampton,no
Abelson,Mr. Samuel,M,30.0,2,24.0,Cherbourg,no
Abelson,Mrs. Samuel (Hannah Wizosky),F,28.0,2,24.0,Cherbourg,yes


*1) Find pairs of passengers who are likely to be twin children: same last name, same age, same embarkation, and age is under 18. Return each pair once, including their last name, first names, age, and embarkation city.*

In [53]:
%%sql
SELECT titanic.last, titanic.first, titanic2.first, titanic.age, titanic.embarked
FROM titanic
INNER JOIN titanic titanic2
    ON titanic.last = titanic2.last
   AND titanic.age = titanic2.age
   AND titanic.embarked = titanic2.embarked
   AND titanic.age < 18
   AND titanic.first < titanic2.first;

 * sqlite://
Done.


last,first,first_1,age,embarked
Baclini,Miss Eugenie,Miss Helene Barbara,0.75,Cherbourg
Calic,Mr. Jovo,Mr. Petar,17.0,Southampton


*2) Which embarkation cities have more than 40 passengers whose age is missing? Reminder: Blanks in SQL tables are given a special value called 'null', and conditions 'A is null' and 'A is not null' can be used in Where clauses to check whether attribute A has the 'null' value.*

In [54]:
%%sql
SELECT embarked, COUNT(*)
FROM titanic
WHERE age IS NULL
GROUP BY embarked
HAVING COUNT(*) > 40;

 * sqlite://
Done.


embarked,COUNT(*)
Queenstown,49
Southampton,90


*3) Find all classes where the average fare paid by passengers in that class was more than twice the overall average or less than half the overall average.*

In [55]:
%%sql
SELECT class
FROM titanic
GROUP BY class
HAVING AVG(fare) > 2 * (SELECT AVG(fare) FROM titanic)
    OR AVG(fare) < 0.5 * (SELECT AVG(fare) FROM titanic);

 * sqlite://
Done.


class
1
3


*4) What is the average number of passengers per last name? Hint: Requires using a subquery in the From clause*

In [56]:
%%sql
SELECT AVG(num_passengers)
FROM (
    SELECT last, COUNT(*) AS num_passengers
    FROM titanic
    GROUP BY last
);

 * sqlite://
Done.


AVG(num_passengers)
1.3358320839580209


*5) List each class and its survival rate, i.e., the fraction of passengers in that class who survived. Hints: Use subqueries in the From clause to compute the number of survivers per class and total passengers per class, and force floating point division by multiplying one operand by 1.0*

In [57]:
%%sql
SELECT survivors.class, survivors.num_survived * 1.0 / totals.num_passengers AS survival_rate
FROM (
    SELECT class, COUNT(*) AS num_survived
    FROM titanic
    WHERE survived = 'yes'
    GROUP BY class
) AS survivors
INNER JOIN (
    SELECT class, COUNT(*) AS num_passengers
    FROM titanic
    GROUP BY class
) AS totals
ON survivors.class = totals.class;

 * sqlite://
Done.


class,survival_rate
1,0.6296296296296297
2,0.47282608695652173
3,0.24236252545824846


*6) Modify your previous query to return the survival rate by gender, i.e., of females and of males.*

In [58]:
%%sql
SELECT survivors.gender, survivors.num_survived * 1.0 / totals.num_passengers AS survival_rate
FROM (
    SELECT gender, COUNT(*) AS num_survived
    FROM titanic
    WHERE survived = 'yes'
    GROUP BY gender
) AS survivors
INNER JOIN (
    SELECT gender, COUNT(*) AS num_passengers
    FROM titanic
    GROUP BY gender
) AS totals
ON survivors.gender = totals.gender;

 * sqlite://
Done.


gender,survival_rate
F,0.7420382165605095
M,0.18890814558058924


*7) Now return the survival rate of children versus adults, i.e., of passengers under age 18 versus those 18 or over (ignoring passengers whose age is missing).*

In [59]:
%%sql
SELECT age_group, num_survived * 1.0 / num_passengers AS survival_rate
FROM (
    SELECT
        CASE
            WHEN age < 18 THEN 'child'
            ELSE 'adult'
        END AS age_group,
        COUNT(*) AS num_passengers,
        SUM(CASE WHEN survived = 'yes' THEN 1 ELSE 0 END) AS num_survived
    FROM titanic
    WHERE age IS NOT NULL
    GROUP BY age_group
);

 * sqlite://
Done.


age_group,survival_rate
adult,0.3810316139767055
child,0.5398230088495575


### <font color = 'green'>**Your Turn Extra - SQL Data Modification on Titanic Data**</font>

In [72]:
# Reload table from CSV file
# NOTE: You may want to reload frequently to reset the data as you
# experiment with modifications
with open('Titanic.csv') as f: Titanic = pd.read_csv(f, index_col=0)
%sql drop table if exists Titanic;
%sql --persist Titanic

 * sqlite://
Done.
 * sqlite://


'Persisted titanic'

In [87]:
%%sql
select * from Titanic

 * sqlite://
Done.


last,first,gender,age,class,fare,embarked,survived
Cardeza,Mr. Thomas Drake Martinez,M,36.0,1,512.33,Cherbourg,yes
Lesurer,Mr. Gustave J,M,35.0,1,512.33,Cherbourg,yes
Ward,Miss Anna,F,35.0,1,512.33,Cherbourg,yes


*1) Subtract 5 from the fare paid by any passenger under the age of 10. Then compute the new average fare. NOTE: You can put multiple SQL statements in one cell separated by a semicolon.*

In [78]:
%%sql
UPDATE Titanic
SET fare = fare - 5
WHERE age < 10;

SELECT AVG(fare) FROM Titanic;

 * sqlite://
62 rows affected.
Done.


AVG(fare)
31.85709315375985


*2) Create a new table called Survivors, containing the last and first names of all passengers who survived. Then count the number of tuples in the new table.*

In [83]:
%%sql
DROP TABLE IF EXISTS Survivors;

CREATE TABLE Survivors AS
SELECT last, first
FROM Titanic
WHERE survived = '1';

 * sqlite://
Done.
Done.


[]

In [84]:
%%sql
SELECT COUNT(*) FROM Survivors;

 * sqlite://
Done.


COUNT(*)
0


*3) In the Titanic table delete all but the highest-paying passengers.*

In [85]:
%%sql
DELETE FROM Titanic
WHERE fare < (SELECT MAX(fare) FROM Titanic);

 * sqlite://
888 rows affected.


[]

In [86]:
%%sql
SELECT * FROM Titanic;

 * sqlite://
Done.


last,first,gender,age,class,fare,embarked,survived
Cardeza,Mr. Thomas Drake Martinez,M,36.0,1,512.33,Cherbourg,yes
Lesurer,Mr. Gustave J,M,35.0,1,512.33,Cherbourg,yes
Ward,Miss Anna,F,35.0,1,512.33,Cherbourg,yes


*4) In what's left of the table after (3), insert a new tuple for yourself. You can decide your class, fare, where you embarked, and whether you survived. Then show the whole table.*

In [94]:
%%sql
INSERT INTO Titanic (last, first, gender, age, class, fare, embarked, survived)
VALUES ('Regadon', 'Marcus', 'Male', 25, 1, 600.0, 'Cherbourg', 'yes');

 * sqlite://
1 rows affected.


[]

In [95]:
%%sql
SELECT * FROM Titanic;

 * sqlite://
Done.


last,first,gender,age,class,fare,embarked,survived
Cardeza,Mr. Thomas Drake Martinez,M,36.0,1,512.33,Cherbourg,yes
Lesurer,Mr. Gustave J,M,35.0,1,512.33,Cherbourg,yes
Ward,Miss Anna,F,35.0,1,512.33,Cherbourg,yes
Regadon,Marcus,Male,25.0,1,600.0,Cherbourg,yes
